In [1]:
# Cell No: 1
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import numpy as np
import time

In [2]:
# Cell No: 2
def load_subset(train_size=10000, test_size=2000):
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()

    x_train = x_train[:train_size].astype("float32") / 255.0
    y_train = to_categorical(y_train[:train_size], 10)

    x_test = x_test[:test_size].astype("float32") / 255.0
    y_test = to_categorical(y_test[:test_size], 10)

    return x_train, y_train, x_test, y_test

In [3]:
# Cell No: 3
def build_custom_cnn(input_shape=(32,32,3), num_classes=10):
    inputs = layers.Input(shape=input_shape)

    # Block 1 (low-level features)
    x = layers.Conv2D(32, (3,3), strides=1, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3,3), strides=1, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 2 (mid-level features)
    x = layers.Conv2D(64, (3,3), strides=1, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3,3), strides=1, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)

    # Block 3 (higher-level features)
    x = layers.Conv2D(128, (3,3), strides=1, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Classifier (low parameter count vs dense-heavy)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

In [4]:
# Cell No: 4
def show_model_summary(model):
    model.summary()

In [5]:
# Cell No: 5
def extract_layer_specs(model):
    print("\n--- Layer Specifications ---\n")

    for layer in model.layers:
        config = layer.get_config()
        params = layer.count_params()

        print(f"Layer: {layer.name}")
        print(f"  Type: {layer.__class__.__name__}")

        if "kernel_size" in config:
            print(f"  Kernel Size: {config.get('kernel_size')}")
        if "strides" in config:
            print(f"  Stride: {config.get('strides')}")
        if "padding" in config:
            print(f"  Padding: {config.get('padding')}")
        if "filters" in config:
            print(f"  Filters: {config.get('filters')}")
        if "activation" in config:
            print(f"  Activation: {config.get('activation')}")

        print(f"  Parameters: {params}\n")

In [6]:
# Cell No: 6
def compile_and_train(model, x_train, y_train):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    start = time.time()

    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=10,
        batch_size=64,
        verbose=1
    )

    end = time.time()

    return history, (end - start)

In [7]:
# Cell No: 7
def evaluate(model, x_test, y_test):
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    return loss, acc

In [8]:
# Cell No: 8
def analyze_model(model, history, training_time):
    total_params = model.count_params()

    print("\n--- Analytical Justification ---\n")

    print(f"Total Parameters: {total_params}")
    print(f"Training Time: {training_time:.2f} sec")

    print(f"Final Train Accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"Final Val Accuracy: {history.history['val_accuracy'][-1]:.4f}")

    gap = history.history['accuracy'][-1] - history.history['val_accuracy'][-1]
    print(f"Generalization Gap: {gap:.4f}")


In [9]:
# Cell No: 9
def main():
    x_train, y_train, x_test, y_test = load_subset()

    model = build_custom_cnn()

    show_model_summary(model)
    extract_layer_specs(model)

    history, training_time = compile_and_train(model, x_train, y_train)

    test_loss, test_acc = evaluate(model, x_test, y_test)

    print("\n--- Test Performance ---")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f}")

    analyze_model(model, history, training_time)

main()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,122 (617.66 KB)

 Trainable params: 157,674 (615.91 KB)

 Non-trainable params: 448 (1.75 KB)


--- Layer Specifications ---

Layer: input_layer
  Type: InputLayer
  Parameters: 0

Layer: conv2d
  Type: Conv2D
  Kernel Size: (3, 3)
  Stride: (1, 1)
  Padding: same
  Filters: 32
  Activation: relu
  Parameters: 896

Layer: batch_normalization
  Type: BatchNormalization
  Parameters: 128

Layer: conv2d_1
  Type: Conv2D
  Kernel Size: (3, 3)
  Stride: (1, 1)
  Padding: same
  Filters: 32
  Activation: relu
  Parameters: 9248

Layer: max_pooling2d
  Type: MaxPooling2D
  Stride: (2, 2)
  Padding: valid
  Parameters: 0

Layer: conv2d_2
  Type: Conv2D
  Kernel Size: (3, 3)
  Stride: (1, 1)
  Padding: same
  Filters: 64
  Activation: relu
  Parameters: 18496

Layer: batch_normalization_1
  Type: BatchNormalization
  Parameters: 256

Layer: conv2d_3
  Type: Conv2D
  Kernel Size: (3, 3)
  Stride: (1, 1)
  Padding: same
  Filters: 64
  Activation: relu
  Parameters: 36928

Layer: max_pooling2d_1
  Type: MaxPooling2D
  Stride: (2, 2)
  Padding: valid
  Parameters: 0

Layer: conv2d_4
  Type: